Unifilar LV Diagram - Sendim
=
This notebook draws the low-voltage single-line diagram (unifilar BT) of UFV Sendim (8 UFVs, 320 inverters, 5637 strings) from Excel tables. It computes where every inverter block, inverter tag, AC cable tag, string tag and module power label goes, and writes AutoCAD scripts (`.scr`) plus an Excel copy of the string table. It doesn't read any DWG.

This is the current Sendim version. Compared with `Unifilar LV optimized.ipynb`:
- Reads the `UFV Sendim - Unifilar BT - 07.05.26.xlsx` workbook.
- Module power is per string (`POT STR` from the `STR` sheet, merged on `TAG`) instead of per inverter.
- Strings are ordered inside each inverter by distance, read from `UFV Sendim - MC LV.xlsx`.
- The AC cable tag Y (`Y cabo`) is computed with a row loop (the first version left most of it `NaN`), and the inverter insert point has its X,Y comma.
- `INV-6.1.18` is included in the 17-string override.

**Workflow overview:**
1. Read the parameter, inverter, string power and string tables, and compute the inverter and inverter tag positions.
2. Compute the AC cable tag positions and write the three inverter scripts.
3. Read the string distances.
4. Build the string table and add distances, tags and module power.
5. Apply the per-PST / per-inverter overrides, the T-C-B exception, and order the strings inside each inverter.
6. Put each string on a slot (row) of its inverter using `PATTERNS`, then write the string tag and module power scripts and the Excel table.

**Input prerequisites:**
- `UFV Sendim - Unifilar BT - 07.05.26.xlsx` (path in the first code cell):
  - `Parametros` sheet: columns C:D, header on row 2. The values are read **by row position** from column D (`df_parameter.iloc[row, 1]`), so the row order must not change. What each row is used for (names inferred from the code):
    - 0 X origin, 1 Y origin, 2 X pitch per switchboard (PCS/QG pair), 3 X pitch between DJ columns, 4 Y pitch between DJs, 5 DJs per column
    - 6 / 7 inverter tag X / Y offset from the inverter
    - 8 not used
    - 9 / 10 string tag X / Y offset from the inverter, 11 string tag Y pitch per slot
    - 12 / 13 module power X / Y offset from the inverter, 14 module power Y pitch per slot
    - 15 AC cable tag X offset, 16 / 17 AC cable tag Y offset for QG 1 / QG 2, 18 AC cable tag Y step per DJ
  - `INV` sheet: columns A:H, header on row 2. The **first column** is the inverter tag (`TAG`, e.g. `INV-1.1.03`), plus `PD`, `QG`, `DJ`, `Dist` (AC cable length, m), `UFV`, `PCS` and `INV`.
  - `STR` sheet: columns A and R, header on row 2: `TAG` (string tag) and `POT STR` (module power). One row per string, otherwise the merge duplicates strings.
  - `Sheet1` sheet: one row per string with `Tag completa` (e.g. `STR-1.1.01.01`), `UFV`, `PST`, `INV`, `STR`, `Type` (T / B / C) and `Type N` (1 / 2 / 3).
- `UFV Sendim - MC LV.xlsx`, `Database` sheet: columns K:L, header on row 2: the string tag with a 4-character prefix in front of `STR-...` (removed with `str[4:]`) and the string distance.
- String tags must be fixed-width `STR-U.P.II.SS` (1-digit UFV and PST, 2-digit INV, e.g. `STR-1.1.01.05`). The inverter tag is cut from characters 4-10 (`INV-1.1.01`) and the PST tag from characters 4-7 (`PST-1.1`). A 2-digit UFV or PST breaks both.

**Target drawing prerequisites** (to run the generated scripts, which use no DWG as input):
- A block named `Inversor` is defined in the drawing and is **not** set to *Scale uniformly*: the insert line answers X scale, Y scale and rotation (`1 1 0`). If the block has attributes, set `ATTREQ = 0`, because the script gives no attribute values.
- The layers `EMF_unifilar_inversor`, `EMF_TAG_inversor`, `CA_Cabos`, `TAG_strings` and `Potência_Módulo` exist. Each script starts with `-LAYER SET <layer>`, which fails on a missing layer.
- MText is written with the current text style (only height, justification, rotation and width are set).

Inputs and inverter positions
=
- `qty_strings = 18` and `pattern = 1` are the defaults for every string. They are overridden per PST / inverter later.
- `inv_step = 9842.2311`: vertical distance between two UFV blocks. UFVs are stacked with `inv_step · (8 − UFV)`, so UFV 8 is at the bottom (offset 0) and UFV 1 at the top.
- `PATTERNS`: slot rows (1-based) for 18 (pattern 1), 17 (pattern 2) and 16 (pattern 3) strings. The gaps (4, 8-9, 13-14, 18, 22-23) separate the MPPT groups.
- `math` is not used.
- Inverter block position:
  - `X = X0 + (2·PCS − 3 + QG) · pitch_switchboard + pitch_DJcolumn · trunc((DJ − 1) / DJs_per_column)`. Each PCS/QG pair gets its own column band (PCS 1/QG 1 → 0, PCS 1/QG 2 → 1, PCS 2/QG 1 → 2, ...), and the DJs wrap into a new column every `DJs_per_column`.
  - `Y = inv_step · (8 − UFV) + Y0 + ((DJ − 1) mod DJs_per_column) · pitch_DJ`. The expression `DJ − ceil(DJ/n)·n + n − 1` is just `(DJ − 1) mod n`.
- `X tag` / `Y tag`: inverter tag position = inverter position + fixed offset (rows 6 / 7).
- `X cabo`: AC cable tag column, one per PCS/QG pair (row 15 offset).
- The table is sorted by `UFV, PCS, QG, DJ` (the order of the script lines).
- `Y cabo` starts at `0.0` and is filled in the next cell.

In [13]:
import time

import pandas as pd

import math

import numpy as np

start = time.time()

qty_strings = 18
pattern = 1

inv_step = 9842.2311

PATTERNS = {
    1: [1,2,3,5,6,7,10,11,12,15,16,17,19,20,21,24,25,26],
    2: [1,2,3,5,6,7,10,11,12,15,16,17,19,20,21,24,25],
    3: [1,2,3,5,6,7,10,11,15,16,17,19,20,21,24,25]
}

with pd.ExcelFile(r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\302 - Zagope - UFV Sendim\01 - Projetos\01 - Em Confecção\02 - Edição\02 - Planilhas\UFV Sendim - Unifilar BT - 07.05.26.xlsx") as xlsx:
    df_parameter = pd.read_excel(xlsx, sheet_name="Parametros", usecols="C:D", header=1)
    df_inv = pd.read_excel(xlsx, sheet_name="INV", usecols="A:H", header=1)
    df_pot = pd.read_excel(xlsx, sheet_name="STR", usecols="A,R", header=1)
    df_type = pd.read_excel(xlsx, sheet_name="Sheet1")


df_inv['X'] = df_parameter.iloc[0, 1]+(df_inv["PCS"]*2-3+df_inv["QG"])*df_parameter.iloc[2, 1]+df_parameter.iloc[3, 1]*np.trunc((df_inv["DJ"]-1)/df_parameter.iloc[5, 1])
df_inv['Y'] = (inv_step * (8 - df_inv['UFV'])) + (df_parameter.iloc[1, 1] + (df_inv['DJ'] - np.ceil(df_inv['DJ'] / df_parameter.iloc[5, 1]) * df_parameter.iloc[5, 1] + df_parameter.iloc[5, 1] - 1.0) * df_parameter.iloc[4, 1])
df_inv['X tag'] = df_inv['X'] + df_parameter.iloc[6, 1]
df_inv['Y tag'] = df_inv['Y'] + df_parameter.iloc[7, 1]
df_inv['X cabo'] = df_parameter.iloc[0, 1]+(df_inv['PCS']*2+df_inv['QG']-3)*df_parameter.iloc[2, 1]+df_parameter.iloc[15, 1]
df_inv['Y cabo'] = 0.0

df_inv = df_inv.sort_values(by=["UFV", "PCS", "QG", "DJ"]).reset_index(drop=True)

print(df_inv)

            TAG  PD  QG  DJ    Dist  UFV  PCS  INV        X           Y  \
0    INV-1.1.03   1   1   1  190.24    1    1    3   5000.0  73895.6177   
1    INV-1.1.04   1   1   2  173.11    1    1    4   5000.0  72364.3677   
2    INV-1.1.05   1   1   3  141.71    1    1    5   5000.0  70833.1177   
3    INV-1.1.09   1   1   4  178.69    1    1    9   6410.0  73895.6177   
4    INV-1.1.11   1   1   5  145.31    1    1   11   6410.0  72364.3677   
..          ...  ..  ..  ..     ...  ...  ...  ...      ...         ...   
315  INV-8.2.11   2   2   6   91.24    8    2   11  31235.0   1937.5000   
316  INV-8.2.20   2   2   7  215.77    8    2   20  32645.0   5000.0000   
317  INV-8.2.18   2   2   8  113.37    8    2   18  32645.0   3468.7500   
318  INV-8.2.14   2   2  10   65.26    8    2   14  34055.0   5000.0000   
319  INV-8.2.13   2   2  11  108.93    8    2   13  34055.0   3468.7500   

       X tag       Y tag   X cabo  Y cabo  
0     4930.0  73595.6177   9540.0     0.0  
1     4930.

AC cable tag positions and inverter scripts
=
- `Y cabo` (AC cable tag Y), row by row:
  - DJ 1 of QG 1 → `inv_step·(8 − UFV) + Y0 + offset_QG1` (row 16); DJ 1 of QG 2 → same with row 17.
  - Other DJs → the DJ 1 value of their QG + `(DJ − 1) · step` (row 18), so the cable tags form one column per QG.
  - `test` is assigned in the loop but never used.
- `TAG Cabo CA` = `CLV-UFV.PCS.INV.(L1-L2-L3) - <Dist>m` (INV zero-padded to 2 digits, length rounded to 0.1 m).
- `Script 1`: `-insert Inversor X,Y 1 1 0` (X scale 1, Y scale 1, rotation 0).
- `Script 2`: MText with the inverter tag (`TAG`), height 30, justification ML, rotation −90.
- `Script 3`: MText with the AC cable tag, height 20, justification TL, rotation 0.
- Writes `Sendim - Script Inversor.scr` (layer `EMF_unifilar_inversor`), `Sendim - Script TAG Inversor.scr` (layer `EMF_TAG_inversor`) and `Sendim - Script Cabos CA.scr` (layer `CA_Cabos`) in `Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar`.

In [14]:
for i in range(len(df_inv)):
    if (df_inv.loc[i, 'QG'] == 1) and (df_inv.loc[i, 'DJ'] == 1):
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[16, 1])
    elif (df_inv.loc[i, 'QG'] == 2) and (df_inv.loc[i, 'DJ'] == 1):
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[17, 1])
    elif df_inv.loc[i, 'QG'] == 1:
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[16, 1]) + (df_inv.loc[i, 'DJ']-1)*df_parameter.iloc[18, 1]
        test = df_inv.loc[i, 'Y cabo']
    else:
        df_inv.loc[i, 'Y cabo'] = (inv_step * (8 - df_inv.loc[i, "UFV"])) + (df_parameter.iloc[1, 1] + df_parameter.iloc[17, 1]) + (df_inv.loc[i, 'DJ']-1)*df_parameter.iloc[18, 1]

df_inv['TAG Cabo CA'] = "CLV-" + df_inv["UFV"].astype(str) + "." + \
    df_inv["PCS"].astype(str) + "." + \
    df_inv['INV'].astype(str).str.zfill(2) + \
    ".(L1-L2-L3) - " + \
    df_inv["Dist"].round(1).astype(str) + "m"

df_inv['Script 1'] = "-insert Inversor " + \
                     df_inv['X'].astype(str) + "," +\
                     df_inv['Y'].astype(str) + \
                     " 1 1 0"

df_inv['Script 2'] = "(command-s \"._mtext\" \""+ df_inv['X tag'].astype(str) + "," + \
                     df_inv['Y tag'].astype(str) + \
                     "\" \"h\" 30  \"j\" \"ML\" \"r\" -90  \"w\" 0 \"" + \
                     df_inv['TAG'].astype(str) + \
                     "\" \"\")"

df_inv['Script 3'] = "(command-s \"._mtext\" \""+ df_inv['X cabo'].astype(str) + "," + \
                     df_inv['Y cabo'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"TL\" \"r\" 0  \"w\" 0 \"" + \
                     df_inv['TAG Cabo CA'].astype(str) + \
                     "\" \"\")"


with open (r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar\Sendim - Script Inversor.scr", 'w') as f:
    f.write('-LAYER\nSET\nEMF_unifilar_inversor\n\n')
    f.write('\n'.join(df_inv["Script 1"].astype(str)) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar\Sendim - Script TAG Inversor.scr", 'w') as f:
    f.write('-LAYER\nSET\nEMF_TAG_inversor\n\n')
    f.write('\n'.join(df_inv["Script 2"].astype(str)) + '\n')

with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar\Sendim - Script Cabos CA.scr", 'w') as f:
    f.write('-LAYER\nSET\nCA_Cabos\n\n')
    f.write('\n'.join(df_inv["Script 3"].astype(str)) + '\n')

String distances
=
Reads the string tag and distance from the `Database` sheet of `UFV Sendim - MC LV.xlsx` (columns K:L) and removes the first 4 characters of the tag, so it matches `STR-U.P.II.SS`. The distance is used to order the strings inside each inverter.

The last row is empty (`NaN`); it doesn't match any string, so it is harmless.

In [15]:
String_Distance = pd.read_excel(r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\302 - Zagope - UFV Sendim\01 - Projetos\01 - Em Confecção\02 - Edição\02 - Planilhas\UFV Sendim - MC LV.xlsx", sheet_name = "Database", header = 1, names = ["TAG", "Distance"], usecols = "K:L")

String_Distance['TAG'] = String_Distance['TAG'].astype(str).str[4:]
print(String_Distance)

                TAG  Distance
0     STR-1.1.01.01     34.77
1     STR-1.1.01.02     25.62
2     STR-1.1.01.03     25.66
3     STR-1.1.01.04     18.11
4     STR-1.1.01.05     18.15
...             ...       ...
5633  STR-8.2.20.14    108.84
5634  STR-8.2.20.15    108.88
5635  STR-8.2.20.16    115.36
5636  STR-8.2.20.17    115.40
5637            NaN       NaN

[5638 rows x 2 columns]


Building the string table
=
- Renames the first `df_inv` column (the inverter tag) to `TAG INV` for the merge with the strings later.
- `df_strings` is built from `Sheet1`, with `Qty. of Strings = 18` and `PD = 1` for all. `PV`, `X`, `Y`, `XX`, `YY` and the script columns start empty.
- Left-merge with `String_Distance` on `TAG` adds `Distance`.

In [16]:
df_inv = df_inv.rename(columns={df_inv.columns[0]: "TAG INV"})

 # Strings TAGs
df_strings = pd.DataFrame({
    'TAG': df_type["Tag completa"],
    'TAG Especial': None,
    'TAG INV': None,
    'TAG PST': None,
    'UFV': df_type["UFV"],
    'PST': df_type["PST"],
    'INV': df_type["INV"],
    'STR': df_type["STR"],
    'Type': df_type["Type"],
    'Type N': df_type["Type N"],
    'Qty. of Strings': qty_strings,
    'PD': pattern,
    'PV': None,
    'X': None,
    'Y': None,
    'XX': None,
    'YY': None,
    'Script TAG String': None,
    'Script Potência': None,
})

df_strings = df_strings.merge(String_Distance, on="TAG", how="left")
print(df_strings)

                TAG TAG Especial TAG INV TAG PST  UFV  PST  INV  STR Type  \
0     STR-1.1.01.01         None    None    None    1    1    1    1    C   
1     STR-1.1.01.02         None    None    None    1    1    1    2    T   
2     STR-1.1.01.03         None    None    None    1    1    1    3    B   
3     STR-1.1.01.04         None    None    None    1    1    1    4    T   
4     STR-1.1.01.05         None    None    None    1    1    1    5    B   
...             ...          ...     ...     ...  ...  ...  ...  ...  ...   
5632  STR-8.2.20.13         None    None    None    8    2   20   13    C   
5633  STR-8.2.20.14         None    None    None    8    2   20   14    T   
5634  STR-8.2.20.15         None    None    None    8    2   20   15    B   
5635  STR-8.2.20.16         None    None    None    8    2   20   16    T   
5636  STR-8.2.20.17         None    None    None    8    2   20   17    B   

      Type N  Qty. of Strings  PD    PV     X     Y    XX    YY  \
0       

Tags and module power
=
- `TAG Especial` = `(Type) STR-...` (e.g. `(C) STR-1.1.01.01`). `TAG INV` / `TAG PST` are cut from the string tag.
- Left-merge with the `STR` sheet on `TAG` adds `POT STR` (module power of each string).

In [17]:

df_strings["TAG Especial"] = "("+df_strings['Type'].astype(str)+") "+df_strings['TAG'].astype(str)
df_strings["TAG INV"] = "INV-" + df_strings["TAG"].astype(str).str.slice(4, 10)
df_strings["TAG PST"] = "PST-" + df_strings["TAG"].astype(str).str.slice(4, 7)
df_strings = df_strings.merge(df_pot, on="TAG", how="left")
print(df_strings)

                TAG       TAG Especial     TAG INV  TAG PST  UFV  PST  INV  \
0     STR-1.1.01.01  (C) STR-1.1.01.01  INV-1.1.01  PST-1.1    1    1    1   
1     STR-1.1.01.02  (T) STR-1.1.01.02  INV-1.1.01  PST-1.1    1    1    1   
2     STR-1.1.01.03  (B) STR-1.1.01.03  INV-1.1.01  PST-1.1    1    1    1   
3     STR-1.1.01.04  (T) STR-1.1.01.04  INV-1.1.01  PST-1.1    1    1    1   
4     STR-1.1.01.05  (B) STR-1.1.01.05  INV-1.1.01  PST-1.1    1    1    1   
...             ...                ...         ...      ...  ...  ...  ...   
5632  STR-8.2.20.13  (C) STR-8.2.20.13  INV-8.2.20  PST-8.2    8    2   20   
5633  STR-8.2.20.14  (T) STR-8.2.20.14  INV-8.2.20  PST-8.2    8    2   20   
5634  STR-8.2.20.15  (B) STR-8.2.20.15  INV-8.2.20  PST-8.2    8    2   20   
5635  STR-8.2.20.16  (T) STR-8.2.20.16  INV-8.2.20  PST-8.2    8    2   20   
5636  STR-8.2.20.17  (B) STR-8.2.20.17  INV-8.2.20  PST-8.2    8    2   20   

      STR Type  Type N  ...  PD    PV     X     Y    XX    YY  

String count / pattern overrides and ordering
=
- Hardcoded Sendim overrides (the inverter overrides run second, so they win):
  - PST-5.2, 7.2, 8.2 → 17 strings, pattern 2. PST-5.1, 7.1 → 16 strings, pattern 3.
  - INV-5.2.11, 5.2.15, 5.2.19 → 16 strings, pattern 3. INV-6.1.18, 6.1.19, 6.1.20 → 17 strings, pattern 2.
  - These must match the real string count of each inverter, because the slot assignment in the next cell depends on it.
- T-C-B exception (from the code comment): in an inverter with **17 strings and exactly one C string**, the B strings get `Type N = 5` and the C string `Type N = 4`. Sorting by `Type N` then gives T-C-B (1-4-5) instead of the usual T-B-C (1-2-3), so the C string shares an MPPT with T strings.
  - The loop looks each row up by `TAG` and only updates the first match, so a duplicated string tag is only fixed once.
- Ordering by distance: `sorting_distance` is `−Distance` for B strings (`Type N` 2 and 5) and `Distance` for the others, so inside each type T / C strings go nearest first and B strings farthest first. `ascending_types` is defined but not used.
- The commented-out block is an alternative sort by the plant's execution order of the PSTs.
- Sorts by `UFV, PST, INV, Type N, sorting_distance` and drops the helper column.

In [18]:
df_strings["Qty. of Strings"] = np.where(
    (df_strings["TAG PST"] == "PST-5.2") | (df_strings["TAG PST"] == "PST-7.2") | (df_strings["TAG PST"] == "PST-8.2"),
    17,
    np.where(
        (df_strings["TAG PST"] == "PST-5.1") | (df_strings["TAG PST"] == "PST-7.1"),
        16,
        df_strings["Qty. of Strings"]
    )
)

df_strings["PD"] = np.where(
    (df_strings["TAG PST"] == "PST-5.2") | (df_strings["TAG PST"] == "PST-7.2") | (df_strings["TAG PST"] == "PST-8.2"),
    2,
    np.where(
        (df_strings["TAG PST"] == "PST-5.1") | (df_strings["TAG PST"] == "PST-7.1"),
        3,
        df_strings["PD"]
    )
)

df_strings["Qty. of Strings"] = np.where(
    (df_strings["TAG INV"] == "INV-5.2.11") | (df_strings["TAG INV"] == "INV-5.2.15") | (df_strings["TAG INV"] == "INV-5.2.19"),
    16,
    np.where(
        (df_strings["TAG INV"] == "INV-6.1.18") | (df_strings["TAG INV"] == "INV-6.1.19") | (df_strings["TAG INV"] == "INV-6.1.20"),
        17,
        df_strings["Qty. of Strings"]
    )
)

df_strings["PD"] = np.where(
    (df_strings["TAG INV"] == "INV-5.2.11") | (df_strings["TAG INV"] == "INV-5.2.15") | (df_strings["TAG INV"] == "INV-5.2.19"),
    3,
    np.where(
        (df_strings["TAG INV"] == "INV-6.1.18") | (df_strings["TAG INV"] == "INV-6.1.19") | (df_strings["TAG INV"] == "INV-6.1.20"),
        2,
        df_strings["PD"]
    )
)

all_tags = df_strings["TAG INV"].unique()
c_counts = (df_strings[df_strings["Type N"] == 3]["TAG INV"].value_counts().reindex(all_tags, fill_value=0)) # Counts how many C-type Strings there are in all inverters

# This gets the exception where the C-string should share the MPPT with the T-string so that less MPPTs have different types of strings. So it changes the Type N for B-strings to 5 and the Type N for C-strings to 4 to order them T-C-B (1-4-5) instead of the usual T-B-C (1-2-3).
for tag in df_strings["TAG"]:
    index = df_strings[df_strings["TAG"] == tag].index[0]
    tag_inv = df_strings.at[index, "TAG INV"]
    c_count = c_counts[tag_inv]
    if df_strings.at[index, "Qty. of Strings"] == 17 and c_count == 1:
        if df_strings.at[index, "Type N"] == 2:
            df_strings.at[index, "Type N"] = 5
        elif df_strings.at[index, "Type N"] == 3:
            df_strings.at[index, "Type N"] = 4

# Define which Type N values should have ascending vs descending Distance
ascending_types = [1,3,4]
descending_types = [2,5]


# Create modified distance for sorting
df_strings['sorting_distance'] = np.where(
    df_strings['Type N'].isin(descending_types),
    -df_strings['Distance'],
    df_strings['Distance']
)

# # Order of Execution in Plant
# custom_order = {
#     '1.3': 1, '2.2': 2, '5.2': 3, '6.1': 4, '5.1': 5,
#     '1.1': 6, '7.1': 7, '1.2': 8, '6.2': 9, '4.2': 10,
#     '2.3': 11, '2.1': 12, '4.1': 13, '7.2': 14, '3.1': 15,
#     '3.2': 16, '8.1': 17, '8.2': 18
# }
#
# # Create a combined column for sorting
# df_strings['UFV_PST'] = df_strings['UFV'].astype(str) + '.' + df_strings['PST'].astype(str)
#
# # Map to custom order
# df_strings['custom_order'] = df_strings['UFV_PST'].map(custom_order)
#
# # Sort using the custom order
# df_strings = df_strings.sort_values(
#     by=["custom_order", "INV", "Type N", "sorting_distance"]
# ).reset_index(drop=True)
#
# # Drop temporary columns
# df_strings = df_strings.drop(['UFV_PST', 'custom_order'], axis=1)

df_strings = df_strings.sort_values(
    by=["UFV", "PST", "INV", "Type N", "sorting_distance"]
).reset_index(drop=True)


df_strings = df_strings.drop("sorting_distance", axis=1)

print(df_strings)

                TAG       TAG Especial     TAG INV  TAG PST  UFV  PST  INV  \
0     STR-1.1.01.06  (T) STR-1.1.01.06  INV-1.1.01  PST-1.1    1    1    1   
1     STR-1.1.01.15  (T) STR-1.1.01.15  INV-1.1.01  PST-1.1    1    1    1   
2     STR-1.1.01.04  (T) STR-1.1.01.04  INV-1.1.01  PST-1.1    1    1    1   
3     STR-1.1.01.11  (T) STR-1.1.01.11  INV-1.1.01  PST-1.1    1    1    1   
4     STR-1.1.01.02  (T) STR-1.1.01.02  INV-1.1.01  PST-1.1    1    1    1   
...             ...                ...         ...      ...  ...  ...  ...   
5632  STR-8.2.20.03  (B) STR-8.2.20.03  INV-8.2.20  PST-8.2    8    2   20   
5633  STR-8.2.20.08  (B) STR-8.2.20.08  INV-8.2.20  PST-8.2    8    2   20   
5634  STR-8.2.20.06  (C) STR-8.2.20.06  INV-8.2.20  PST-8.2    8    2   20   
5635  STR-8.2.20.01  (C) STR-8.2.20.01  INV-8.2.20  PST-8.2    8    2   20   
5636  STR-8.2.20.13  (C) STR-8.2.20.13  INV-8.2.20  PST-8.2    8    2   20   

      STR Type  Type N  ...  PD    PV     X     Y    XX    YY  

Slots, string scripts and output files
=
- `Position in group` = running count **per `PD`**, and `PV = PATTERNS[PD][position mod len(PATTERNS[PD])]` is the slot (row) of the string next to its inverter.
  - Warning: the counter runs per pattern over the whole plant, not per inverter. It only lines up if every inverter has exactly as many strings as its pattern has slots (18 / 17 / 16). One inverter with a different count shifts the slots of every following inverter with that pattern.
- Left-merge with the inverter `X`, `Y` on `TAG INV`. `df_strings` already has empty `X` / `Y` columns, so pandas names them `X_x` / `Y_x` (empty) and `X_y` / `Y_y` (inverter position).
  - `X_x` / `Y_x` are then overwritten with the **string tag** position: `X_inv + row 9`, `Y_inv + row 10 + row 11 · (PV − 1)`.
  - `XX` / `YY` hold the **module power** position: `X_inv + row 12`, `Y_inv + row 13 + row 14 · (PV − 1)`.
- `Script TAG String`: MText of `TAG Especial`, height 20, justification MR. `Script Potência`: MText of the module power, height 20, justification MC.
- Module power comes from `POT STR`.
- Outputs in `Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar`: `Sendim - Diagrama Unifilar BT.xlsx` (the full string table), `Sendim - tag string.scr` (layer `TAG_strings`) and `Sendim - potencia.scr` (layer `Potência_Módulo`).
- Prints the time elapsed since the first cell.

In [19]:
df_strings["Position in group"] = df_strings.groupby("PD").cumcount()

# adicionar C,T,B
df_strings['PV'] = df_strings.apply(
    lambda row: PATTERNS[row['PD']][row['Position in group'] % len(PATTERNS[row['PD']])],
    axis=1
)

df_strings = df_strings.merge(df_inv[["TAG INV", "X", "Y"]], on="TAG INV", how="left")

df_strings['X_x'] = df_strings['X_y'].astype(float)+df_parameter.iloc[9,1]
df_strings['Y_x'] = df_strings['Y_y'].astype(float)+df_parameter.iloc[10,1]+df_parameter.iloc[11, 1]*(df_strings["PV"].astype(float)-1)
df_strings['XX'] = df_strings['X_y'].astype(float)+df_parameter.iloc[12,1]
df_strings['YY'] = df_strings['Y_y'].astype(float)+df_parameter.iloc[13,1]+df_parameter.iloc[14, 1]*(df_strings["PV"].astype(float)-1)

df_strings['Script TAG String'] = "(command-s \"._mtext\" \""+ df_strings['X_x'].astype(str) + "," + \
                     df_strings['Y_x'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"MR\" \"r\" 0  \"w\" 0 \"" + \
                     df_strings['TAG Especial'].astype(str) + \
                     "\" \"\")"

df_strings['Script Potência'] = "(command-s \"._mtext\" \""+ df_strings['XX'].astype(str) + "," + \
                     df_strings['YY'].astype(str) + \
                     "\" \"h\" 20  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     df_strings['POT STR'].astype(str) + \
                     "\" \"\")"

df_strings.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar\Sendim - Diagrama Unifilar BT.xlsx")
print(df_strings)

with open (r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar\Sendim - tag string.scr", 'w') as f:
    f.write('-LAYER\nSET\nTAG_strings\n\n')
    f.write('\n'.join(df_strings["Script TAG String"].astype(str)) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Diagrama Unifilar\Sendim - potencia.scr", 'w') as f:
    f.write('-LAYER\nSET\nPotência_Módulo\n\n')
    f.write('\n'.join(df_strings["Script Potência"].astype(str)) + '\n')


end = time.time()
elapsed = end - start
print(elapsed)

                TAG       TAG Especial     TAG INV  TAG PST  UFV  PST  INV  \
0     STR-1.1.01.06  (T) STR-1.1.01.06  INV-1.1.01  PST-1.1    1    1    1   
1     STR-1.1.01.15  (T) STR-1.1.01.15  INV-1.1.01  PST-1.1    1    1    1   
2     STR-1.1.01.04  (T) STR-1.1.01.04  INV-1.1.01  PST-1.1    1    1    1   
3     STR-1.1.01.11  (T) STR-1.1.01.11  INV-1.1.01  PST-1.1    1    1    1   
4     STR-1.1.01.02  (T) STR-1.1.01.02  INV-1.1.01  PST-1.1    1    1    1   
...             ...                ...         ...      ...  ...  ...  ...   
5632  STR-8.2.20.03  (B) STR-8.2.20.03  INV-8.2.20  PST-8.2    8    2   20   
5633  STR-8.2.20.08  (B) STR-8.2.20.08  INV-8.2.20  PST-8.2    8    2   20   
5634  STR-8.2.20.06  (C) STR-8.2.20.06  INV-8.2.20  PST-8.2    8    2   20   
5635  STR-8.2.20.01  (C) STR-8.2.20.01  INV-8.2.20  PST-8.2    8    2   20   
5636  STR-8.2.20.13  (C) STR-8.2.20.13  INV-8.2.20  PST-8.2    8    2   20   

      STR Type  Type N  ...         Y_x       XX          YY  \